# Wednesday — Wake Word Training (Kaggle Notebooks)

## Before you run this

**1. Create the notebook.** On kaggle.com → **Code** → **New Notebook**.
Either paste these cells in, or use *File → Upload Notebook* with this
`.ipynb` file.

**2. Settings (right sidebar):**
- **Accelerator → GPU T4 x2** — do this *before*
  running any cell; changing it later restarts the session and you lose
  progress.
- **Internet → On** — required for `git clone`, `pip install`, and every
  download in this notebook. Off by default on some notebook types.

**3. Session limits:** ~30 GPU hours/week, sessions up to ~9–12 hours. This
whole pipeline is roughly 1–1.5 hours of active compute plus ~20–40 min of
downloads — comfortably one sitting, no need to plan around multi-session
persistence.

**4. Disk:** `/kaggle/working` auto-saves but is capped at **20GB** — too
small for the ~30GB+ of scratch data (ACAV, FMA, generated clips) this
pipeline uses. Everything large goes in `/kaggle/temp` instead (bigger, but
**wiped when the session ends** — that's fine, it's all disposable
intermediate data). Only the final `.onnx` model and training-history JSON
get copied into `/kaggle/working` at the end, so **click "Save Version"
when you're done** — that's what actually persists them for download.


## 1. Install — apt (root, no sudo needed) + pip, skipping what's already present

In [ ]:
import sys, os, shutil, subprocess
from pathlib import Path
from importlib import metadata as _im

print(f"Running as: {os.environ.get('USER', subprocess.run(['whoami'], capture_output=True, text=True).stdout.strip())}")

# Native tools. Kaggle notebooks run as root, so plain apt-get works with no
# sudo — unlike a lot of other hosted-notebook platforms. Check first since
# Kaggle's base image often already has these (saves a minute or two).
def apt_ensure(binary, apt_pkgs):
    if shutil.which(binary):
        print(f"  already present: {binary}")
        return
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq"] + apt_pkgs, check=True)
    print(f"  installed via apt: {apt_pkgs}")

apt_ensure("ffmpeg", ["ffmpeg"])
apt_ensure("espeak-ng", ["espeak-ng", "espeak-ng-data", "libespeak-ng-dev"])

def ensure(pip_spec, import_name=None):
    import_name = import_name or pip_spec.split("==")[0]
    try:
        __import__(import_name)
        print(f"  already present: {import_name}")
        return
    except ImportError:
        pass
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_spec], check=True)
    print(f"  installed: {pip_spec}")

def ensure_version(pip_spec, dist_name, ok):
    try:
        v = _im.version(dist_name)
        if ok(v):
            print(f"  OK {dist_name} {v} already satisfies constraint")
            return
        print(f"  {dist_name} {v} does not satisfy constraint - reinstalling")
    except _im.PackageNotFoundError:
        pass
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_spec], check=True)
    print(f"  installed: {pip_spec}")

from packaging.version import Version

for pip_spec, import_name in [
    ("piper-phonemize-cross", "piper_phonemize"),
    ("webrtcvad", "webrtcvad"),
    ("mutagen==1.47.0", "mutagen"),
    ("torchinfo", "torchinfo"),
    ("torchmetrics", "torchmetrics"),
    ("pyyaml", "yaml"),
    ("tqdm", "tqdm"),
    ("datasets", "datasets"),
    ("soundfile", "soundfile"),
    ("audiomentations", "audiomentations"),
    ("torch_audiomentations", "torch_audiomentations"),
    ("pronouncing", "pronouncing"),
    ("onnxruntime", "onnxruntime"),
    ("onnx", "onnx"),
    ("speechbrain", "speechbrain"),
    ("acoustics", "acoustics"),     # no Windows wheel; builds fine here via apt's gcc
    ("scipy", "scipy"),
    ("requests", "requests"),
    ("huggingface_hub", "huggingface_hub"),
]:
    ensure(pip_spec, import_name)

# Version-pinned (2026 `datasets` breakage - see top markdown cell)
ensure_version("pyarrow<15.0.0", "pyarrow", lambda v: Version(v) < Version("15.0.0"))
ensure_version("fsspec<2024.1.0", "fsspec", lambda v: Version(v) < Version("2024.1.0"))

# piper-tts LAST, --no-deps, so it can't downgrade/clobber anything above
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "piper-tts"], check=True)

import torch
print(f"\ntorch: {torch.__version__}  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU visible. Check Settings -> Accelerator -> GPU T4 x2,")
    print("then Session -> Restart & Run All (changing the accelerator requires a restart).")
try:
    import torchaudio
    print(f"torchaudio: {torchaudio.__version__}")
except ImportError:
    print("torchaudio missing - installing without touching torch (--no-deps)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "torchaudio"], check=True)
    import torchaudio
    print(f"torchaudio: {torchaudio.__version__}")


In [ ]:
# Fail-fast smoke test for the TTS phonemizer - cheap insurance before the
# 10-15 minute clip-generation step later.
from piper_phonemize import phonemize_espeak
sample = phonemize_espeak("hello wednesday", "en-us")
print("piper_phonemize OK, sample phonemes:", sample)


## 2. Working directory — scratch on /kaggle/temp, persisted output on /kaggle/working

In [ ]:
BASE_DIR = Path("/kaggle/temp/wwtrain")
BASE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(BASE_DIR)

OUT_DIR = Path("/kaggle/working")  # 20GB cap, auto-saved on "Save Version"

usage = shutil.disk_usage("/kaggle/temp")
print(f"Working dir: {BASE_DIR}")
print(f"Free space on /kaggle/temp: {usage.free/1e9:.1f} GB")
if usage.free / 1e9 < 30:
    print("WARNING: the full pipeline needs ~30GB+ of scratch space (17GB ACAV")
    print("+ ~8GB FMA zip + extracted/derived files). If this is tight, trim")
    print("n_samples in Section 11's config, or delete ACAV_FULL right after")
    print("Section 10 subsamples it (the notebook already does this).")


## 3. Clone repos + download model (self-healing)

In [ ]:
import requests
from tqdm.auto import tqdm

def run(cmd, cwd=None):
    print("  $", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

def download(url, dest: Path, chunk=4 * 1024 * 1024, resume=False):
    dest.parent.mkdir(parents=True, exist_ok=True)
    resume_byte = dest.stat().st_size if (resume and dest.exists()) else 0
    headers = {"Range": f"bytes={resume_byte}-"} if resume_byte else {}
    mode = "ab" if resume_byte else "wb"
    with requests.get(url, stream=True, headers=headers, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0)) + resume_byte
        with open(dest, mode) as f, tqdm(total=total, initial=resume_byte, unit="B",
                                          unit_scale=True, unit_divisor=1024,
                                          desc=dest.name) as pbar:
            for c in r.iter_content(chunk_size=chunk):
                if c:
                    f.write(c)
                    pbar.update(len(c))

PSG_DIR = BASE_DIR / "piper-sample-generator"
PSG_PIN = "1a8c49bd29b3a132721086ee88f2253f788594a8^"
PSG_GS = PSG_DIR / "generate_samples.py"
if not PSG_GS.exists():
    print("  piper-sample-generator missing or broken - re-cloning")
    shutil.rmtree(PSG_DIR, ignore_errors=True)
    run(["git", "clone", "-q", "https://github.com/rhasspy/piper-sample-generator", str(PSG_DIR)])
run(["git", "fetch", "-q", "--all"], cwd=PSG_DIR)
run(["git", "checkout", "-q", PSG_PIN], cwd=PSG_DIR)
assert PSG_GS.exists(), "piper-sample-generator pin did not yield generate_samples.py"
print(f"  OK piper-sample-generator pinned: {PSG_GS} ({PSG_GS.stat().st_size} bytes)")

PIPER_MODEL = PSG_DIR / "models" / "en_US-libritts_r-medium.pt"
PIPER_MODEL_URL = "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"
if not PIPER_MODEL.exists() or PIPER_MODEL.stat().st_size < 100_000_000:
    print("  libritts model missing/partial - downloading (~200 MB)")
    download(PIPER_MODEL_URL, PIPER_MODEL)
assert PIPER_MODEL.stat().st_size > 100_000_000, "libritts model download failed"
print(f"  OK libritts model: {PIPER_MODEL.stat().st_size/1e6:.0f} MB")

OWW_DIR = BASE_DIR / "openwakeword"
OWW_TRAIN = OWW_DIR / "openwakeword" / "train.py"
if not OWW_TRAIN.exists():
    print("  openwakeword missing or gutted - re-cloning")
    shutil.rmtree(OWW_DIR, ignore_errors=True)
    run(["git", "clone", "-q", "https://github.com/dscripka/openwakeword", str(OWW_DIR)])
    run([sys.executable, "-m", "pip", "install", "-q", "-e", str(OWW_DIR)])
assert OWW_TRAIN.exists(), "openwakeword train.py missing after clone"
print(f"  OK openwakeword: {OWW_TRAIN} ({OWW_TRAIN.stat().st_size/1e3:.0f} KB)")

if str(OWW_DIR) not in sys.path:
    sys.path.insert(0, str(OWW_DIR))
for _m in list(sys.modules):
    if _m.startswith("openwakeword"):
        del sys.modules[_m]
import openwakeword
assert openwakeword.__file__ is not None, "openwakeword loaded as a namespace package"
print(f"  OK openwakeword importable: {openwakeword.__file__}")


## 4. Apply runtime patches (idempotent, pure Python)

In [ ]:
import importlib

import torch_audiomentations
io_path = Path(torch_audiomentations.__file__).parent / "utils" / "io.py"
text = io_path.read_text(encoding="utf-8")
patched = text.replace('torchaudio.set_audio_backend("soundfile")', "pass  # patched")
if patched != text:
    io_path.write_text(patched, encoding="utf-8")
    print(f"  Patch A applied: {io_path}")
else:
    print(f"  Patch A: nothing to change: {io_path}")

src = PSG_DIR / "generate_samples.py"
dst = OWW_DIR / "openwakeword" / "generate_samples.py"
if not dst.exists() or dst.stat().st_size != src.stat().st_size:
    shutil.copy2(src, dst)
print(f"  Patch B (generate_samples copy): {dst}")

os.environ["HF_HUB_ETAG_TIMEOUT"] = "120"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
import huggingface_hub.constants as hfc
for attr in ["DEFAULT_ETAG_TIMEOUT", "DEFAULT_DOWNLOAD_TIMEOUT",
             "HF_HUB_ETAG_TIMEOUT", "HF_HUB_DOWNLOAD_TIMEOUT"]:
    if hasattr(hfc, attr):
        setattr(hfc, attr, 120)
print("  Patch C (HF Hub timeouts -> 120s)")

init_path = Path(torchaudio.__file__)
SHIM_MARKER = "# --- PATCH: info() shim for torchaudio 2.x ---"
content = init_path.read_text(encoding="utf-8")
if SHIM_MARKER not in content:
    shim = (
        f"\n\n{SHIM_MARKER}\n"
        "def info(file_path, *args, **kwargs):\n"
        "    import soundfile as _sf\n"
        "    si = _sf.info(str(file_path))\n"
        '    return type("_TorchaudioInfo", (), {\n'
        '        "num_frames": si.frames, "sample_rate": si.samplerate,\n'
        '        "num_channels": si.channels, "bits_per_sample": 16,\n'
        '        "encoding": "PCM_S",\n'
        "    })()\n"
    )
    init_path.write_text(content + shim, encoding="utf-8")
    importlib.reload(torchaudio)
print(f"  Patch D (torchaudio.info shim): {hasattr(torchaudio, 'info')}")

target = PSG_DIR / "generate_samples.py"
text = target.read_text(encoding="utf-8")
old = "model: Union[str, Path],"
new = f"model: Union[str, Path] = {str(PIPER_MODEL)!r},"
if old in text:
    target.write_text(text.replace(old, new), encoding="utf-8")
    shutil.copy2(target, OWW_DIR / "openwakeword" / "generate_samples.py")
    print("  Patch E applied (generate_samples model arg default)")
else:
    print("  Patch E: target line not found - check manually")

train_py = OWW_DIR / "openwakeword" / "train.py"
text = train_py.read_text(encoding="utf-8")
old_line = "val_predictions = self.model(x_val)"
new_line = "val_predictions = self.model(x_val.float())"
if old_line in text and new_line not in text:
    train_py.write_text(text.replace(old_line, new_line), encoding="utf-8")
    print("  Patch F applied (train.py val dtype cast)")
else:
    print("  Patch F: nothing to change")

print("\nAll patches applied (idempotent).")


## 5. Pre-flight — every dep imports + every external file exists

In [ ]:
import importlib

expected = {
    "piper-sample-generator/generate_samples.py": PSG_DIR / "generate_samples.py",
    "openwakeword/generate_samples.py": OWW_DIR / "openwakeword" / "generate_samples.py",
    "libritts model": PIPER_MODEL,
    "openwakeword/train.py": OWW_DIR / "openwakeword" / "train.py",
    "openwakeword/data.py": OWW_DIR / "openwakeword" / "data.py",
    "openwakeword/utils.py": OWW_DIR / "openwakeword" / "utils.py",
}
missing_files = []
for name, p in expected.items():
    if p.exists():
        size = p.stat().st_size
        unit = "MB" if size > 1e6 else "KB"
        denom = 1e6 if unit == "MB" else 1e3
        print(f"  OK {name}: {size/denom:.1f} {unit}")
    else:
        missing_files.append((name, p))
        print(f"  MISSING: {name} ({p})")

missing_mods = []
for mod in ["torch", "torchinfo", "torchmetrics", "scipy", "numpy", "tqdm",
            "yaml", "mutagen", "pronouncing", "torchaudio", "audiomentations",
            "torch_audiomentations", "speechbrain", "acoustics", "onnx",
            "onnxruntime", "soundfile", "requests", "huggingface_hub",
            "piper_phonemize", "piper", "openwakeword", "webrtcvad"]:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing_mods.append((mod, str(e)))
        print(f"  IMPORT FAIL: {mod} ({type(e).__name__}: {e})")

try:
    from openwakeword.data import generate_adversarial_texts, augment_clips, mmap_batch_generator
    from openwakeword.utils import compute_features_from_generator, AudioFeatures
    print("  OK openwakeword.data + .utils dry-import OK")
except Exception as e:
    print(f"  openwakeword internal import: {type(e).__name__}: {e}")
    missing_mods.append(("openwakeword.data/utils", str(e)))

print(f"\n  CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'}")
if missing_files or missing_mods:
    raise RuntimeError(f"Pre-flight failed - {len(missing_files)} missing files, {len(missing_mods)} import errors.")
print("\n  Pre-flight PASSED. Safe to proceed.")


## 6. Download openwakeword shared models (mel + embedding)

In [ ]:
models_dir = OWW_DIR / "openwakeword" / "resources" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
BASE_URL = "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1"
for fname in ["embedding_model.onnx", "embedding_model.tflite",
              "melspectrogram.onnx", "melspectrogram.tflite"]:
    out = models_dir / fname
    if out.exists() and out.stat().st_size > 1000:
        print(f"  cached: {fname} ({out.stat().st_size/1e3:.0f} KB)")
    else:
        download(f"{BASE_URL}/{fname}", out)
        print(f"  OK downloaded: {fname} ({out.stat().st_size/1e3:.0f} KB)")


## 7. MIT impulse responses → 16-kHz WAVs (~1 min)

In [ ]:
import time
import numpy as np
import scipy.io.wavfile as wavfile
from huggingface_hub import snapshot_download
import datasets

rir_dir = BASE_DIR / "mit_rirs"
rir_dir.mkdir(parents=True, exist_ok=True)
if len([f for f in rir_dir.iterdir() if f.suffix == ".wav"]) >= 250:
    print(f"  cached: {len(list(rir_dir.iterdir()))} MIT IR WAVs")
else:
    for i in range(6):
        try:
            snapshot_download(repo_id="davidscripka/MIT_environmental_impulse_responses",
                               repo_type="dataset", etag_timeout=120, max_workers=4)
            break
        except Exception as e:
            wait = 15 * (i + 1)
            print(f"  snapshot_download attempt {i+1}/6: {type(e).__name__}: {str(e)[:80]} - sleep {wait}s")
            time.sleep(wait)
    for attempt in range(4):
        try:
            rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses",
                                                 split="train", streaming=False)
            break
        except Exception as e:
            print(f"  load_dataset attempt {attempt+1}: {type(e).__name__}: {str(e)[:120]}")
            time.sleep(20 * (attempt + 1))
    else:
        raise RuntimeError("load_dataset failed for MIT IRs")
    n = 0
    for i, row in enumerate(rir_dataset):
        out = rir_dir / f"{i:04d}.wav"
        if out.exists():
            continue
        audio = row["audio"]
        sr = audio["sampling_rate"]
        arr = np.asarray(audio["array"])
        if sr != 16000:
            from scipy.signal import resample_poly
            arr = resample_poly(arr, 16000, sr)
        arr = (arr * 32767).clip(-32768, 32767).astype(np.int16)
        wavfile.write(str(out), 16000, arr)
        n += 1
    print(f"  OK wrote {n} new WAVs (total {len(list(rir_dir.iterdir()))})")


## 8. Download FMA + ACAV features (~10–20 min)

Real code cell here — in the original Colab notebook this was accidentally
typed as **markdown**, so it silently never ran on a fresh runtime.

In [ ]:
import zipfile
os.chdir(BASE_DIR)

ACAV_FULL = BASE_DIR / "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
ACAV_URL = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
if ACAV_FULL.exists() and ACAV_FULL.stat().st_size > 16_000_000_000:
    print(f"  cached: ACAV {ACAV_FULL.stat().st_size/1e9:.1f} GB")
else:
    print(f"  ACAV: {'resuming' if ACAV_FULL.exists() else 'fetching'} (~17 GB)")
    download(ACAV_URL, ACAV_FULL, resume=True)
    print(f"  OK ACAV: {ACAV_FULL.stat().st_size/1e9:.1f} GB")

FMA_DIR = BASE_DIR / "fma"
FMA_ZIP = BASE_DIR / "fma_small.zip"
FMA_DIR.mkdir(parents=True, exist_ok=True)
if FMA_DIR.is_dir() and len(list(FMA_DIR.iterdir())) > 100:
    print(f"  cached: FMA {len(list(FMA_DIR.iterdir()))} entries")
else:
    fma_url = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
    print(f"  FMA: {'resuming' if FMA_ZIP.exists() else 'fetching'} (~8 GB)")
    download(fma_url, FMA_ZIP, resume=True)
    print("  extracting FMA...")
    with zipfile.ZipFile(FMA_ZIP) as zf:
        for name in tqdm(zf.namelist(), desc="extract", unit="f"):
            zf.extract(name, FMA_DIR)
    FMA_ZIP.unlink()
    print(f"  OK FMA extracted: {len(list(FMA_DIR.iterdir()))} entries")


## 9. FMA MP3s → 16-kHz mono WAVs (~5 min)

In [ ]:
import glob

fma_wav_dir = BASE_DIR / "fma_wav"
fma_wav_dir.mkdir(parents=True, exist_ok=True)
n_existing = len(list(fma_wav_dir.iterdir()))
if n_existing >= 1500:
    print(f"  cached: {n_existing} FMA WAVs")
else:
    mp3s = glob.glob(str(FMA_DIR / "**" / "*.mp3"), recursive=True)
    if len(mp3s) < 100:
        raise RuntimeError(f"FMA download incomplete - only {len(mp3s)} MP3s found")
    existing = {p.name for p in fma_wav_dir.iterdir()}
    n_target = min(1500, len(mp3s))
    to_convert = [(i, mp3s[i]) for i in range(n_target) if f"{i:05d}.wav" not in existing]
    print(f"  Converting {len(to_convert)} MP3s -> 16kHz mono WAVs (skipping {n_target - len(to_convert)} cached)")
    for i, mp3 in tqdm(to_convert):
        out = fma_wav_dir / f"{i:05d}.wav"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", mp3,
                        "-ac", "1", "-ar", "16000", str(out)], check=False)
    print(f"  OK {len(list(fma_wav_dir.iterdir()))} FMA WAVs ready")


## 10. Subsample ACAV — 1.7 GB train + 170 MB val

In [ ]:
TRAIN_DST = BASE_DIR / "acav_train_subset.npy"
VAL_DST = BASE_DIR / "acav_val_subset.npy"
if TRAIN_DST.exists() and VAL_DST.exists():
    train_arr = np.load(TRAIN_DST, mmap_mode="r")
    val_arr = np.load(VAL_DST, mmap_mode="r")
    print(f"  cached: train={train_arr.shape} ({train_arr.nbytes/1e9:.2f} GB), "
          f"val={val_arr.shape} ({val_arr.nbytes/1e6:.0f} MB)")
else:
    if not ACAV_FULL.exists():
        raise RuntimeError(f"ACAV source missing: {ACAV_FULL} - re-run Section 8")
    arr = np.load(ACAV_FULL, mmap_mode="r")
    print(f"  source: {arr.shape}, dtype={arr.dtype}")
    n_total = len(arr)
    n_train = n_total // 10
    n_val_rows = n_total // 100
    train_chunk = np.array(arr[:n_train])
    np.save(TRAIN_DST, train_chunk)
    print(f"  OK train: {train_chunk.shape} ({train_chunk.nbytes/1e9:.2f} GB)")
    del train_chunk
    val_chunk = np.array(arr[n_train:n_train + n_val_rows])
    val_flat = val_chunk.reshape(-1, val_chunk.shape[-1])
    np.save(VAL_DST, val_flat)
    print(f"  OK val: {val_flat.shape} ({val_flat.nbytes/1e6:.0f} MB)")
    del val_chunk, val_flat
    ACAV_FULL.unlink() 
    print("  removed 17 GB original")


## 11. Build training config

In [ ]:
import yaml

TARGET_PHRASE = ["hey wednesday", "hello wednesday", "hi wednesday"] 
MODEL_NAME = "wednesday"
OUTPUT_DIR = BASE_DIR / f"{MODEL_NAME}_output"

config = {
    "target_phrase": TARGET_PHRASE,
    "model_name": MODEL_NAME,
    "custom_negative_phrases": [],
    "n_samples": 6000,    
    "n_samples_val": 1000,
    "tts_batch_size": 50,
    "piper_sample_generator_path": str(PSG_DIR),
    "augmentation_rounds": 1,
    "augmentation_batch_size": 16,
    "steps": 20000,
    "max_negative_weight": 1500,
    "target_accuracy": 0.7,   
    "target_recall": 0.5,     
    "target_false_positives_per_hour": 0.5,
    "batch_size": 128,
    "learning_rate": 1e-4,
    "model_type": "dnn",
    "layer_dim": 128,
    "layer_size": 128,
    "n_blocks": 1,
    "model_input_shape": [16, 96],
    "n_classes": 1,
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
    "background_paths": [str(BASE_DIR / "fma_wav")],
    "background_paths_duplication_rate": [1],
    "rir_paths": [str(BASE_DIR / "mit_rirs")],
    "false_positive_validation_data_path": str(BASE_DIR / "acav_val_subset.npy"),
    "feature_data_files": {"ACAV100M_sample": str(BASE_DIR / "acav_train_subset.npy")},
    "output_dir": str(OUTPUT_DIR),
    "tflite_export": False,
    "onnx_export": True,
    "positive_clips_train_dir": str(OUTPUT_DIR / MODEL_NAME / "positive_train"),
    "positive_clips_test_dir": str(OUTPUT_DIR / MODEL_NAME / "positive_test"),
    "negative_clips_train_dir": str(OUTPUT_DIR / MODEL_NAME / "negative_train"),
    "negative_clips_test_dir": str(OUTPUT_DIR / MODEL_NAME / "negative_test"),
    "feature_save_dir": str(OUTPUT_DIR / MODEL_NAME),
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = BASE_DIR / "my_model.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.dump(config, f, sort_keys=False)

print(f"  target_phrase: {config['target_phrase']}")
print(f"  n_samples:     {config['n_samples']}")
print(f"  config written to: {CONFIG_PATH}")


## 12. Generate Piper TTS clips (~10–15 min)

In [ ]:
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

dirs = {
    "positive_train": (Path(cfg["positive_clips_train_dir"]), int(cfg["n_samples"] * 0.75)),
    "positive_test": (Path(cfg["positive_clips_test_dir"]), int(cfg["n_samples_val"] * 0.75)),
    "negative_train": (Path(cfg["negative_clips_train_dir"]), int(cfg["n_samples"] * 0.75)),
    "negative_test": (Path(cfg["negative_clips_test_dir"]), int(cfg["n_samples_val"] * 0.75)),
}
status = []
for name, (path, expected) in dirs.items():
    n = len(list(path.iterdir())) if path.is_dir() else 0
    status.append((name, n, expected, n >= expected))
all_full = all(ok for _, _, _, ok in status)
for name, n, expected, ok in status:
    print(f"  {'PASS' if ok else 'MISS'} {name}: {n} clips (expect >={expected})")

if all_full:
    print("\n  cached: all 4 clip dirs already populated")
else:
    train_py = OWW_DIR / "openwakeword" / "train.py"
    subprocess.run([sys.executable, str(train_py),
                    "--training_config", str(CONFIG_PATH),
                    "--generate_clips"], check=True)
    for name, (path, expected) in dirs.items():
        n = len(list(path.iterdir())) if path.is_dir() else 0
        assert n >= expected, f"generate_clips left {name} with only {n} clips (expected >={expected})"
    print("\n  PASS all 4 clip dirs now populated")


## 13. Resample TTS clips 22050 → 16000 Hz

In [ ]:
import soundfile as sf
from scipy.signal import resample_poly

TARGET_SR = 16000
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
ROOT = Path(cfg["output_dir"])
wav_dirs = sorted({p.parent for p in ROOT.rglob("*.wav")})
for d in wav_dirs:
    files = [f for f in d.iterdir() if f.suffix == ".wav"]
    if not files:
        continue
    sr_counts = {}
    for f in files[:20]:
        sr = sf.info(str(f)).samplerate
        sr_counts[sr] = sr_counts.get(sr, 0) + 1
    if all(k == TARGET_SR for k in sr_counts):
        print(f"  ok {d}: {len(files)} files at 16 kHz (skip)")
        continue
    print(f"  resampling {d}: {len(files)} files, SRs {sr_counts}")
    n = 0
    for f in tqdm(files, desc=d.name or str(d), leave=False):
        data, sr = sf.read(str(f))
        if sr == TARGET_SR:
            continue
        new_data = resample_poly(data.astype("float32"), TARGET_SR, sr)
        sf.write(str(f), new_data, TARGET_SR)
        n += 1
    print(f"  ok {d}: resampled {n}/{len(files)}")

for f in ROOT.rglob("*.npy"):
    f.unlink()
print("\n  cleared stale features")


## 14. Augment + featurise (~10 min)

In [ ]:
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
FEAT = Path(cfg["feature_save_dir"])
needed = ["positive_features_train.npy", "negative_features_train.npy",
          "positive_features_test.npy", "negative_features_test.npy"]
if all((FEAT / n).exists() for n in needed):
    print("  cached: all 4 feature .npy files exist")
else:
    train_py = OWW_DIR / "openwakeword" / "train.py"
    subprocess.run([sys.executable, str(train_py),
                    "--training_config", str(CONFIG_PATH),
                    "--augment_clips"], check=True)
for f in sorted(FEAT.glob("*.npy")):
    print(f"  {f}: {f.stat().st_size/1e6:.1f} MB")
for n in needed:
    assert (FEAT / n).exists(), f"augment didn't produce {n}"
print("  PASS all 4 feature files present")


## 15. Hand-rolled trainer (~30–40 min on Kaggle's T4 GPU)


In [ ]:
import copy, math, time
import torch.nn as nn
import torch.optim as optim

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
FEAT = Path(cfg["feature_save_dir"])

pos_train = torch.from_numpy(np.load(FEAT / "positive_features_train.npy").astype(np.float32)).to(DEVICE)
neg_train = torch.from_numpy(np.load(FEAT / "negative_features_train.npy").astype(np.float32)).to(DEVICE)
pos_test = torch.from_numpy(np.load(FEAT / "positive_features_test.npy").astype(np.float32)).to(DEVICE)
neg_test = torch.from_numpy(np.load(FEAT / "negative_features_test.npy").astype(np.float32)).to(DEVICE)
print(f"  pos_train={tuple(pos_train.shape)}  neg_train={tuple(neg_train.shape)}")

acav_train_np = np.load(cfg["feature_data_files"]["ACAV100M_sample"], mmap_mode="r")
acav_val_np = np.load(cfg["false_positive_validation_data_path"])
M = acav_val_np.shape[0]
val_listen_hours = M * 0.08 / 3600.0
n_win_val = M - 16
acav_val_windows = np.lib.stride_tricks.sliding_window_view(acav_val_np, (16, 96))[:, 0, :, :]
acav_val_windows = np.ascontiguousarray(acav_val_windows.astype(np.float32))
VAL_BATCH = 4096


class WakewordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(16 * 96, 128)
        self.layernorm1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 1)

    def forward(self, x):
        return self.layer2(self.relu1(self.layernorm1(self.layer1(self.flatten(x)))))


model = WakewordModel().to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(reduction="none")
TOTAL_STEPS = cfg["steps"]
MAX_NEG_W = cfg["max_negative_weight"]
TARGET_FP_PER_HR = cfg["target_false_positives_per_hour"]
THRESH = 0.5
history = {"val_recall": [], "val_accuracy": [], "val_fp_per_hour": [], "val_n_fp": [], "loss": []}
best_models = []
B_POS, B_ANEG, B_ACAV = 32, 32, 64


def random_acav_window_batch(k):
    N, T, F = acav_train_np.shape
    rows = np.random.randint(0, N, size=k)
    starts = np.random.randint(0, T - 16 + 1, size=k)
    out = np.empty((k, 16, F), dtype=np.float32)
    for i, (r, s) in enumerate(zip(rows, starts)):
        out[i] = acav_train_np[r, s:s + 16, :].astype(np.float32)
    return out


def build_batch():
    p_idx = torch.randint(0, pos_train.shape[0], (B_POS,), device=DEVICE)
    aneg_idx = torch.randint(0, neg_train.shape[0], (B_ANEG,), device=DEVICE)
    p, an = pos_train[p_idx], neg_train[aneg_idx]
    acav = torch.from_numpy(random_acav_window_batch(B_ACAV)).to(DEVICE)
    x = torch.cat([p, an, acav], dim=0)
    y = torch.cat([torch.ones(B_POS, device=DEVICE), torch.zeros(B_ANEG + B_ACAV, device=DEVICE)])
    return x, y


@torch.no_grad()
def validate(step_label):
    model.eval()
    p_preds = torch.sigmoid(model(pos_test)).squeeze(-1)
    n_preds = torch.sigmoid(model(neg_test)).squeeze(-1)
    recall = (p_preds >= THRESH).float().mean().item()
    accuracy = (((p_preds >= THRESH).sum() + (n_preds < THRESH).sum()).item()
                / (pos_test.shape[0] + neg_test.shape[0]))
    n_fp = 0
    for i in range(0, n_win_val, VAL_BATCH):
        chunk = torch.from_numpy(acav_val_windows[i:i + VAL_BATCH]).to(DEVICE)
        n_fp += (torch.sigmoid(model(chunk)).squeeze(-1) >= THRESH).sum().item()
    fp_per_hour = n_fp / max(val_listen_hours, 1e-6)
    history["val_recall"].append(recall)
    history["val_accuracy"].append(accuracy)
    history["val_fp_per_hour"].append(fp_per_hour)
    history["val_n_fp"].append(n_fp)
    save = False
    if len(history["val_n_fp"]) >= 3:
        fp_p50 = np.percentile(history["val_n_fp"], 50)
        rc_p5 = np.percentile(history["val_recall"], 5)
        if n_fp <= fp_p50 and recall >= rc_p5:
            best_models.append((copy.deepcopy(model.state_dict()),
                                {"val_recall": recall, "val_accuracy": accuracy,
                                 "val_fp_per_hour": fp_per_hour, "val_n_fp": n_fp}))
            save = True
    print(f"  [{step_label}] recall={recall:.3f}  acc={accuracy:.3f}  "
          f"fp/hr={fp_per_hour:.2f}  saved={'+' if save else '-'}", flush=True)
    model.train()


def run_stage(stage_idx, n_steps, lr, max_neg_w, val_window_frac=1.0):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    weight_schedule = np.linspace(1.0, max_neg_w, n_steps)
    val_start = int(n_steps * (1.0 - val_window_frac))
    val_steps = set(np.linspace(val_start, n_steps - 1, 20).astype(int))
    warmup = max(1, n_steps // 5)
    hold = n_steps // 3
    accumulated = []
    print(f"\n=== Stage {stage_idx}: {n_steps} steps, lr={lr}, max_neg_w={max_neg_w} ===", flush=True)
    t0 = time.time()
    for step in range(n_steps):
        if step < warmup:
            lr_now = lr * (step + 1) / warmup
        elif step < warmup + hold:
            lr_now = lr
        else:
            decay_t = (step - warmup - hold) / max(1, n_steps - warmup - hold)
            lr_now = lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_t)))
        for pg in optimizer.param_groups:
            pg["lr"] = lr_now
        x, y = build_batch()
        logits = model(x).squeeze(-1)
        preds = torch.sigmoid(logits)
        keep = ((y == 0) & (preds >= 0.001)) | ((y == 1) & (preds < 0.999))
        if keep.sum() == 0:
            if step in val_steps:
                validate(f"stage{stage_idx} step {step}/{n_steps}")
            continue
        kept_logits = logits[keep]
        kept_y = y[keep]
        neg_w = weight_schedule[step]
        w = torch.where(kept_y > 0.5, torch.tensor(1.0, device=DEVICE),
                         torch.tensor(neg_w, device=DEVICE, dtype=torch.float32))
        accumulated.append((kept_logits, kept_y, w))
        if sum(t[0].shape[0] for t in accumulated) >= 128:
            cat_logits = torch.cat([t[0] for t in accumulated])
            cat_y = torch.cat([t[1] for t in accumulated])
            cat_w = torch.cat([t[2] for t in accumulated])
            loss = (loss_fn(cat_logits, cat_y) * cat_w).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            history["loss"].append(loss.item())
            accumulated.clear()
        if step in val_steps:
            elapsed = (time.time() - t0) / 60
            validate(f"stage{stage_idx} step {step}/{n_steps} ({elapsed:.1f}m, lr={lr_now:.6f}, neg_w={neg_w:.0f})")
    print(f"  Stage {stage_idx} done in {(time.time()-t0)/60:.1f} min", flush=True)


stage1_steps = TOTAL_STEPS
stage2_steps = max(2000, TOTAL_STEPS // 10)
stage3_steps = max(2000, TOTAL_STEPS // 10)
max_neg_w_now = MAX_NEG_W
run_stage(1, stage1_steps, lr=1e-4, max_neg_w=max_neg_w_now, val_window_frac=0.25)
if history["val_fp_per_hour"] and min(history["val_fp_per_hour"]) > TARGET_FP_PER_HR:
    max_neg_w_now *= 2
    print(f"\n[adapt] stage1 best FP/hr > target -> max_neg_w doubled to {max_neg_w_now}")
run_stage(2, stage2_steps, lr=1e-5, max_neg_w=max_neg_w_now, val_window_frac=1.0)
if history["val_fp_per_hour"] and min(history["val_fp_per_hour"]) > TARGET_FP_PER_HR:
    max_neg_w_now *= 2
    print(f"\n[adapt] stage2 best FP/hr > target -> max_neg_w doubled to {max_neg_w_now}")
run_stage(3, stage3_steps, lr=1e-6, max_neg_w=max_neg_w_now, val_window_frac=1.0)
print(f"\n=== Training done. {len(best_models)} checkpoints saved. ===")
print(f"  best val_fp_per_hour: {min(history['val_fp_per_hour']):.2f}")
print(f"  best val_recall:      {max(history['val_recall']):.3f}")
print(f"  best val_accuracy:    {max(history['val_accuracy']):.3f}")


## 16. Ensemble best checkpoints + ONNX export

The final model and training history are copied to `/kaggle/working` — the
**only** location that persists. Click **"Save Version" → "Save & Run All
(Commit)"** when this finishes, then grab the files from the notebook's
**Output** tab.

In [ ]:
import json

if not best_models:
    print("  no saved checkpoints - using current model state")
    final_state = {k: v.clone() for k, v in model.state_dict().items()}
else:
    accs = [s["val_accuracy"] for _, s in best_models]
    rcs = [s["val_recall"] for _, s in best_models]
    fps = [s["val_fp_per_hour"] for _, s in best_models]
    acc_p90 = np.percentile(accs, 90)
    rc_p90 = np.percentile(rcs, 90)
    fp_p10 = np.percentile(fps, 10)
    qualified = [(sd, sc) for sd, sc in best_models
                 if sc["val_accuracy"] >= acc_p90 and sc["val_recall"] >= rc_p90
                 and sc["val_fp_per_hour"] <= fp_p10]
    if not qualified:
        sorted_models = sorted(best_models, key=lambda t: (t[1]["val_fp_per_hour"], -t[1]["val_recall"]))
        qualified = [sorted_models[0]]
        print("  no checkpoint qualified - fell back to single best")
    keys = qualified[0][0].keys()
    final_state = {k: torch.stack([sd[k].float() for sd, _ in qualified]).mean(dim=0) for k in keys}
    print(f"  ensemble of {len(qualified)}/{len(best_models)} qualified checkpoints")

model.load_state_dict(final_state)
model.eval()

class WakewordExportable(torch.nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base

    def forward(self, x):
        return torch.sigmoid(self.base(x))

export_model = WakewordExportable(model).to(DEVICE).eval()
dummy = torch.randn(1, 16, 96, device=DEVICE)

# Write straight to /kaggle/working so it's in the persisted Output.
onnx_path = OUT_DIR / f"{cfg['model_name']}.onnx"
torch.onnx.export(export_model, dummy, str(onnx_path),
                   input_names=["onnx::Flatten_0"], output_names=["output"],
                   dynamic_axes={"onnx::Flatten_0": {0: "batch"}, "output": {0: "batch"}},
                   opset_version=14, dynamo=False)
print(f"  OK wrote {onnx_path} ({onnx_path.stat().st_size/1e3:.0f} KB)")

import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path))
with torch.no_grad():
    p_test_np = pos_test.cpu().numpy()
    p_scores = sess.run(None, {sess.get_inputs()[0].name: p_test_np})[0].flatten()
    print(f"  positive test set: mean={p_scores.mean():.3f}, recall@0.5={(p_scores>=0.5).mean():.3f}")
print(f"  ACAV val FP/hour at 0.5: {history['val_fp_per_hour'][-1]:.2f}")

metrics_path = OUT_DIR / f"{cfg['model_name']}_training_history.json"
with open(metrics_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"  training history saved: {metrics_path}")

print(f"\nDONE. Files in /kaggle/working (persisted on Save Version):")
print(f"  {onnx_path.name}")
print(f"  {metrics_path.name}")
print("\nNow click 'Save Version' -> 'Save & Run All (Commit)', then download")
print("both files from the notebook's Output tab and drop the .onnx into your")
print(f"Wednesday project, e.g.: models/wakeword/{cfg['model_name']}.onnx")
